# Training Failure Analysis

Inspect the persisted guardrail surfaces for the baseline training stack. The notebook is designed
for triaging vocabulary, unknown-token, structural, and pathological-document failure modes without
rerunning tokenization, training, or evaluation.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from motifml.evaluation.structural_checks import StructuralCheckReport
from motifml.evaluation.unknown_tokens import UnknownTokenUsageReport
from motifml.pipelines.tokenization.models import coerce_vocabulary_stats_report
from motifml.training.model_input import TokenizedDocumentRow
from motifml.training.model_input_stats import (
    coerce_model_input_stats_report,
    render_model_input_stats_markdown,
)
from motifml.training.token_codec import (
    coerce_frozen_vocabulary,
    decode_token_ids_to_strings,
)

pd.set_option("display.max_colwidth", 120)

repo_root = Path.cwd()
fixture_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_FIXTURE_ROOT",
        repo_root / "tests" / "fixtures" / "training",
    )
).resolve()
smoke_bundle_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_SMOKE_BUNDLE_ROOT",
        fixture_root / "smoke_bundle",
    )
).resolve()


def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


metrics_payload = load_json(smoke_bundle_root / "metrics.json")
validation_metrics = metrics_payload["splits"]["validation"]
split_unknowns = UnknownTokenUsageReport(**validation_metrics["unknown_token_usage"])
generated_unknowns = UnknownTokenUsageReport(
    **validation_metrics["generated_unknown_token_usage"]
)
structural_report = StructuralCheckReport(**validation_metrics["structural"])
vocab_stats = coerce_vocabulary_stats_report(
    load_json(fixture_root / "vocab_stats.json")
)
model_input_stats = coerce_model_input_stats_report(
    load_json(fixture_root / "model_input_stats.json")
)
vocabulary = coerce_frozen_vocabulary(load_json(fixture_root / "vocabulary.json"))
representative_rows = [
    TokenizedDocumentRow.from_row_dict(load_json(path))
    for path in sorted((fixture_root / "representative_rows").rglob("*.row.json"))
    if path.is_file()
]
row_unknown_summary = []
for row in representative_rows:
    decoded = decode_token_ids_to_strings(row.token_ids, vocabulary=vocabulary)
    unk_count = sum(1 for token in decoded if token == "<unk>")
    row_unknown_summary.append(
        {
            "relative_path": row.relative_path,
            "split": row.split.value,
            "token_count": row.token_count,
            "unk_token_count": unk_count,
            "unk_rate": 0.0 if row.token_count == 0 else unk_count / row.token_count,
        }
    )
row_unknown_summary.sort(key=lambda item: (-item["unk_rate"], item["relative_path"]))
first_failure = structural_report.failures[0] if structural_report.failures else None

display(
    Markdown(
        "## Failure Mode Overview\n\n"
        f"- Validation `<unk>` Rate: `{split_unknowns.unk_rate:.6f}` ({split_unknowns.unk_token_count}/{split_unknowns.token_count})\n"
        f"- Generated `<unk>` Rate: `{generated_unknowns.unk_rate:.6f}` ({generated_unknowns.unk_token_count}/{generated_unknowns.token_count})\n"
        f"- Vocabulary Guardrails Passed: `{vocab_stats.guardrails.passed}`\n"
        f"- Structural Failure Count: `{len(structural_report.failures)}`\n"
        f"- Worst Document Token Count: `{model_input_stats.worst_offending_documents[0].token_count}`\n"
    )
)

In [ ]:
display(
    Markdown(
        "## Unknown Token Review\n\n"
        f"- Highest Representative Row `<unk>` Rate: `{row_unknown_summary[0]['unk_rate']:.6f}`\n"
        f"- Highest Representative Row Document: `{row_unknown_summary[0]['relative_path']}`\n"
        f"- Validation Threshold Passed: `{split_unknowns.passed}`\n"
    )
)
display(pd.DataFrame(row_unknown_summary))

In [ ]:
display(
    Markdown(
        "## Vocabulary Coverage\n\n"
        f"- Vocabulary Size: `{vocab_stats.vocabulary_size}`\n"
        f"- Top Token: `{vocab_stats.guardrails.top_token}`\n"
        f"- Top Token Fraction: `{vocab_stats.guardrails.top_token_fraction:.6f}`\n"
        f"- Missing Required Families: `{', '.join(vocab_stats.guardrails.missing_required_token_families) or 'none'}`\n"
    )
)
display(
    pd.DataFrame(
        [
            {
                "family": entry.family,
                "vocabulary_size": entry.vocabulary_size,
                "token_count": entry.token_count,
            }
            for entry in vocab_stats.token_family_coverage
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"token": entry.token, "count": entry.count}
            for entry in vocab_stats.top_tokens
        ]
    )
)

In [ ]:
failure_rows = [
    {
        "sequence_id": failure.sequence_id,
        "check_name": failure.check_name,
        "token_index": failure.token_index,
        "token": failure.token,
        "message": failure.message,
    }
    for failure in structural_report.failures
]
display(
    Markdown(
        "## Structural Drift\n\n"
        f"- Valid Transition Rate: `{structural_report.valid_transition_rate:.6f}`\n"
        f"- Duration Distribution TV: `{structural_report.duration_distribution_total_variation:.6f}`\n"
        f"- Pitch Out-of-Range Fraction: `{structural_report.out_of_range_pitch_fraction:.6f}`\n"
        f"- First Failure Check: `{first_failure.check_name if first_failure is not None else 'none'}`\n"
        f"- First Failure Document: `{first_failure.sequence_id if first_failure is not None else 'none'}`\n"
    )
)
display(pd.DataFrame(failure_rows))

In [ ]:
display(
    Markdown(
        "## Document Pathologies\n\n"
        + render_model_input_stats_markdown(model_input_stats)
    )
)
display(
    pd.DataFrame(
        [
            {
                "relative_path": entry.relative_path,
                "split": entry.split.value,
                "shard_id": entry.shard_id,
                "token_count": entry.token_count,
                "oversized": entry.exceeds_oversized_threshold,
            }
            for entry in model_input_stats.worst_offending_documents
        ]
    )
)